In [1]:
from utils import load_all_games_csv, get_teams, basic_win_prob_for_et, predict_lr, evaluate_elo_prob_func
from elos import EloTracker
from scipy.special import expit
import numpy as np

# Model Selection

This notebook will compare several different methods to estimate win probabilities from Elo ratings, and possibly home advantage, travel distance, and rest days.

## Get all Games

In [2]:
all_games = load_all_games_csv(preprocess=True)
#all_games = all_games[(all_games['season'] >= 1990) & (all_games['season'] < 2000)]
all_games.head()

/Users/lancehendricks/Documents/College Coding/ML/Elo Ratings/analysis/src/utils/misc_utils.py:33: DtypeWarning: Columns (10,11,13,17,19,20,21,27,28) have mixed types. Specify dtype option on import or set low_memory=False.
  all_games = pd.read_csv(data_dir / filename)


,visteam,hometeam,site,date,number,starttime,daynight,innings,tiebreaker,usedh,...,vispitcherminusteamrgs,homefirstpitchergameofseason,visfirstpitchergameofseason,homelastkwinpct,vislastkwinpct,adjustedvisrestdays,adjustedhomerestdays,adjustedhomedistancetraveled,adjustedvisdistancetraveled,adjustedmarginofvictory
gid,,,,,,,,,,,,,,,,,,,,,
CIN189804150,CL4,CIN,CIN05,18980415,0.0,0:00PM,day,NaN,NaN,False,...,0.0,True,True,NaN,NaN,3.0,3.0,0.0,6.081473,1.000000
LS3189804150,PIT,LS3,LOU03,18980415,0.0,0:00PM,day,NaN,NaN,False,...,0.0,True,True,NaN,NaN,3.0,3.0,0.0,7.010260,2.645751
SLN189804150,CHN,SLN,STL05,18980415,0.0,0:00PM,day,NaN,NaN,False,...,0.0,True,True,NaN,NaN,3.0,3.0,0.0,6.375653,1.000000
BLN189804160,WSN,BLN,BAL07,18980416,0.0,0:00PM,day,NaN,NaN,False,...,0.0,True,True,NaN,NaN,3.0,3.0,0.0,3.278478,2.236068
CIN189804160,CL4,CIN,CIN05,18980416,0.0,0:00PM,day,NaN,NaN,False,...,-25.9,True,True,1.0,0.0,1.0,1.0,0.0,0.000000,1.414214


## Evaluate Simple Probability model

In [3]:
bce, accuracy = evaluate_elo_prob_func(all_games, basic_win_prob_for_et, K=3, margin_of_victory_column='adjustedmarginofvictory', skip_first_n=0)
print(f"BCE: {bce}")
print(f"Accuracy: {accuracy}")

BCE: 0.6794833765075686
Accuracy: 0.5651411521689236


## With +28 Adjustment for Home Team

In [4]:
bce, accuracy = evaluate_elo_prob_func(all_games, lambda home_elo, away_elo, game_info: basic_win_prob_for_et(home_elo + 28, away_elo, game_info), K=2, margin_of_victory_column='adjustedmarginofvictory', skip_first_n=0)
print(f"BCE: {bce}")
print(f"Accuracy: {accuracy}")

BCE: 0.6764792626452175
Accuracy: 0.5720817075969704


## With + 1.9% Adjustment for Home Team

In [5]:
bce, accuracy = evaluate_elo_prob_func(all_games, lambda home_elo, away_elo, game_info: basic_win_prob_for_et(home_elo*1.019, away_elo, game_info), K=2, margin_of_victory_column='adjustedmarginofvictory', skip_first_n=0)
print(f"BCE: {bce}")
print(f"Accuracy: {accuracy}")

BCE: 0.6764717794260093
Accuracy: 0.5718108790452145


## With Logistic Regression

In [3]:
# First need to fit using some Elos - use the initial basic probability func.

df_to_fit = all_games.copy()
et = EloTracker(K=3, elo_prob_func=basic_win_prob_for_et, margin_of_victory_column='adjustedmarginofvictory')
et.add_history(df_to_fit, add_elos_to_df=True)

df_to_fit = df_to_fit.dropna(subset=['adjustedhomedistancetraveled', 'adjustedvisdistancetraveled', 'adjustedhomerestdays', 'adjustedvisrestdays', 'homepitcherminusteamrgs', 'vispitcherminusteamrgs'])


In [4]:
df_to_fit['elodiff'] = df_to_fit['viselobefore'] - df_to_fit['homeelobefore']
df_to_fit['restdiff'] = df_to_fit['adjustedvisrestdays'] - df_to_fit['adjustedhomerestdays']
df_to_fit['distancediff'] = df_to_fit['adjustedvisdistancetraveled'] - df_to_fit['adjustedhomedistancetraveled']
df_to_fit['homediff'] =  0 - 1
df_to_fit['pitcherdiff'] = df_to_fit['vispitcherminusteamrgs'] - df_to_fit['homepitcherminusteamrgs']
# df_to_fit['momentumdiff'] = df_to_fit['vismomentum'] - df_to_fit['homemomentum']

#features = ['elodiff', 'distancediff', 'restdiff']
features = ['elodiff', 'homediff', 'restdiff', 'distancediff', 'pitcherdiff']
X = df_to_fit[features].to_numpy()
y = df_to_fit['homewon'].astype(int).to_numpy().reshape(-1,1)

s = -np.log(10) / 400

In [5]:
# Fit via GD
w = np.zeros((5,1))
w[0,0] = s # Becomes 1 once dividing by s

step = 0.01 # Slightly higher for small gradients
iterations = 10000

for _ in range(iterations):

    z = X @ w
    y_hat = expit(z)
    
    w_grad = (1/X.shape[0]) * X.T @ (y_hat - y)
    
    w_grad[0,0] = 0
    
    #print(w_grad)
    
    w = w - step*w_grad
    
w

array([[-0.00575646],
       [-0.15879801],
       [-0.02854395],
       [ 0.00147698],
       [-0.00909051]])

In [6]:
# Convert back to interpretable coefficients for individual Elo adjustments
w = (1/s) * w
w

array([[ 1.        ],
       [27.58603899],
       [ 4.95859171],
       [-0.25657825],
       [ 1.57918407]])

In [7]:
bce, accuracy = evaluate_elo_prob_func(all_games, lambda home_elo, away_elo, game: predict_lr(home_elo, away_elo, game, w), K=3, margin_of_victory_column='adjustedmarginofvictory', skip_first_n=0)
print(f"BCE: {bce}")
print(f"Accuracy: {accuracy}")

BCE: 0.6753916623142618
Accuracy: 0.574514574248336
